In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


sales = spark.table("globalretail.silver.fact_sales")
products = spark.table("globalretail.silver.dim_products").filter(col("is_current") == True)
stores = spark.table("globalretail.silver.dim_store")
dates = spark.table("globalretail.silver.dim_date")

mart = (
    sales.join(products.select("product_sk", "category", "subcategory", "brand"), on="product_sk", how="left")
    .join(stores.select("store_sk", "store_name", "region", "country_code"), on="store_sk", how="left")
    .join(dates.select("date_key", "calendar_date", "month_name", "quarter_num", "year_num"), on="date_key", how="left")
    .groupBy("calendar_date", "month_name", "quarter_num", "year_num", "region", "country_code", "channel", "category", "subcategory")
    .agg(
        countDistinct("order_id").alias("order_count"),
        sum("quantity").alias("units_sold"),
        sum("net_amount").alias("net_sales_amount"),
        sum("usd_net_amount").alias("usd_net_sales_amount"),
        sum("gross_margin_amount").alias("gross_margin_amount"),
        avg("net_amount").alias("avg_line_value")
    )
)

mart.write.mode("overwrite").saveAsTable("globalretail.gold.sales_gold")
